# Homework 1 torch的基本操作与梯度下降算法

Author: Lianjie (Marc) Shi  
请使用 `torch` 的基本操作完成下述操作：

In [19]:
import torch

1．设定参数，样本量 $n=1000$ ，变量维度 $p=10$ ，回归系数 $\beta_0 \sim U_{10}(0,1)$ ．

In [20]:
n = 1000
p = 10
torch.manual_seed(42)
beta0 = torch.rand(p, 1) # beta0 ~ Uniform(0,1), shape [p,1]

2. 生成单次模拟数据：设计矩阵 $X=\left(X_1, \ldots, X_n\right)^{\top} \in \mathbb{R}^{1000 \times 10}$ ，其中 $X_i$ 相互独立，且满足 $X_i \sim N_{10}\left(0, I_{10}\right)$ ；误差项向量 $\varepsilon=\left(\varepsilon_1, \ldots, \varepsilon_n\right) \in \mathbb{R}^n$ ，其中 $\varepsilon_i$ 相互独立，且满足 $\varepsilon_i \sim N(0,0.16)$

In [21]:
X = torch.randn(n, p) # X: (n,p), 每行X_i ~ N(0, I_p)

eps = torch.randn(n, 1) * 0.4 # epsilon: N(0,0.16) → std=0.4

3．根据线性回归模型生成因变量 $Y=X \beta_0+\varepsilon \in \mathbb{R}^n$ ．

In [22]:
Y = X @ beta0 + eps

4．（i）请利用 torch 中关于张量的操作，计算如下的最小二乘估计：

$$
\widehat{\beta}_{o l s}=\left(X^{\top} X\right)^{-1}\left(X^{\top} Y\right)
$$

In [23]:
XtX = X.T @ X
XtY = X.T @ Y
beta_ols = torch.linalg.inv(XtX) @ XtY

print("OLS estimate beta_ols:\n", beta_ols)

OLS estimate beta_ols:
 tensor([[0.8804],
        [0.9210],
        [0.3948],
        [0.9515],
        [0.3773],
        [0.5962],
        [0.2484],
        [0.7744],
        [0.9577],
        [0.1301]])


（ii）请利用 torch 中关于张量的操作，实现梯度下降算法计算最小二乘估计：

$$
\widehat{\beta}_{g d}^{(t+1)}=\widehat{\beta}_{g d}^{(t)}-\alpha g^{(t+1)}, \quad g^{(t)}=-\frac{2}{n} X^{\top} Y+\frac{2}{n}\left(X^{\top} X\right) \widehat{\beta}_{g d}^{(t)},
$$

其中 $t=0,1, \ldots, T, T$ 是一个充分大的迭代次数．$\alpha$ 是自己确定的学习率．$\alpha$ 和 $T$ 的选取由同学们自行确定．算法可以选择运行固定迭代次数终止或根据收敛性条件终止．

In [24]:
alpha = 0.001  # 学习率
T = 10000     # 迭代次数
beta_gd = torch.zeros(p, 1) # 初始值

for t in range(T): #选择固定迭代次数：迭代T次即终止
    # g(t) = -2/n * X^T Y + 2/n * (X^T X) beta
    g = (-2.0 / n) * X.T @ Y + (2.0 / n) * (X.T @ X) @ beta_gd
    beta_gd = beta_gd - alpha * g

print("GD estimate beta_gd:\n", beta_gd)

GD estimate beta_gd:
 tensor([[0.8804],
        [0.9210],
        [0.3948],
        [0.9515],
        [0.3773],
        [0.5962],
        [0.2484],
        [0.7744],
        [0.9577],
        [0.1301]])


（iii）请利用 torch 中关于张量的操作，实现小批量梯度下降算法计算最小二乘估计：

$$
\widehat{\beta}_{m g d}^{(t, m+1)}=\widehat{\beta}_{g d}^{(t, m)}-\alpha g^{(t, m+1)}, \quad \widehat{\beta}_{m g d}^{(t+1,0)}=\widehat{\beta}_{m g d}^{(t, M)},
$$

其中 $t=0,1, \ldots, T, T$ 是一个充分大的迭代次数．$\alpha$ 是自己确定的学习率．$m=0,1, \ldots, M-1, M$ 表示小批量的个数．

In [ ]:
batch_size = 50 #200/50=4个batch，
alpha_mb = 0.001 #学习率保持不变
T = 200 # 轮数
beta_mgd = torch.zeros(p,1)
for epoch in range(T):
    perm = torch.randperm(n) #permute the indices 
    X_shuffle = X[perm,:]
    Y_shuffle = Y[perm,:]
    for start in range(0, n, batch_size):
        end = start + batch_size
        Xb = X_shuffle[start:end, :]
        Yb = Y_shuffle[start:end, :]
        nb = Xb.shape[0]
        g_mb = (-2.0 / nb) * Xb.T @ Yb + (2.0 / nb) * (Xb.T @ Xb) @ beta_mgd
        beta_mgd = beta_mgd - alpha_mb * g_mb
print("Mini-batch GD estimate beta_mgd:\n", beta_mgd)


Mini-batch GD estimate beta_mgd:
 tensor([[0.8799],
        [0.9207],
        [0.3944],
        [0.9509],
        [0.3771],
        [0.5959],
        [0.2478],
        [0.7739],
        [0.9572],
        [0.1300]])


5．重复步骤 $2-4$ ，进行 100 次重复实验，假设这个过程中记录了 3 种估计量各 100 个．

In [ ]:
beta_ols_list = []
beta_gd_list = []
beta_mgd_list = []

n_repeat = 100
for rep in range(n_repeat):
    
    X = torch.randn(n, p) # X_i ~ N10(0,I10)
    eps = torch.randn(n,1) * 0.4 # eps~N(0,0.16)，标准差0.4

    Y = X @ beta0 + eps

    XtX = X.T @ X
    XtY = X.T @ Y
    beta_ols = torch.linalg.inv(XtX) @ XtY

    alpha = 0.001
    T_gd = 10000
    beta_gd = torch.zeros(p,1)
    for t in range(T_gd):
        g = (-2.0 / n) * X.T @ Y + (2.0 / n) * (X.T @ X) @ beta_gd
        beta_gd = beta_gd - alpha * g

    batch_size = 50
    M = n // batch_size # M就是小批量个数，1000/50=20
    alpha_mb = 0.001
    T_mb = 200
    beta_mgd = torch.zeros(p,1)
    for epoch in range(T_mb):
        perm = torch.randperm(n)
        X_shuffle = X[perm,:]
        Y_shuffle = Y[perm,:]
        for m in range(M):
            start = m * batch_size
            end = start + batch_size
            Xb = X_shuffle[start:end,:]
            Yb = Y_shuffle[start:end,:]
            nb = Xb.shape[0]
            g_mb = (-2.0 / nb) * Xb.T @ Yb + (2.0 / nb) * (Xb.T @ Xb) @ beta_mgd
            beta_mgd = beta_mgd - alpha_mb * g_mb


    beta_ols_list.append(beta_ols.detach().clone())
    beta_gd_list.append(beta_gd.detach().clone())
    beta_mgd_list.append(beta_mgd.detach().clone())

6．根据PPT上的计算公式，计算三种估计量的RMSE估计值，谈谈你的看法。

（i）对于梯度下降算法来说，学习率的选取对结果和训练过程（迭代次数）有什么关系？

（ii）对于小批量梯度下降算法来说，$M$ 的个数与学习率 $\alpha$ 如何影响结果和训练过程？

In [ ]:

beta_ols_all = torch.stack(beta_ols_list)
beta_gd_all = torch.stack(beta_gd_list)
beta_mgd_all = torch.stack(beta_mgd_list)

# PPT RMSE公式 sqrt( 1/B * sum ||beta_hat - beta0||^2 )
B = n_repeat
norm2_ols = torch.sum((beta_ols_all - beta0)**2, dim=1)
norm2_gd = torch.sum((beta_gd_all - beta0)**2, dim=1)
norm2_mgd = torch.sum((beta_mgd_all - beta0)**2, dim=1)

rmse_ols = torch.sqrt(torch.mean(norm2_ols))
rmse_gd = torch.sqrt(torch.mean(norm2_gd))
rmse_mgd = torch.sqrt(torch.mean(norm2_mgd))

print(f"RMSE OLS: {rmse_ols.item():.6f}")
print(f"RMSE Full GD: {rmse_gd.item():.6f}")
print(f"RMSE Mini-batch GD: {rmse_mgd.item():.6f}")


RMSE OLS: 0.038958
RMSE Full GD: 0.038958
RMSE Mini-batch GD: 0.038966


简答题：

通过RMSE的结果我们可以看到： OLS 是闭式解析解，不存在迭代误差，RMSE 最低；全量梯度下降 GD 在迭代次数充分大时可以收敛到 OLS 解，RMSE 与 OLS 十分接近；小批量梯度下降 MBGD 利用部分样本计算梯度，梯度存在随机噪声，在最优解附近持续震荡，因此 RMSE 略高于 OLS 和全量 GD。

(i): 学习率 $\alpha$ 控制每一次 GD 更新的步长：

$$
\beta^{(t+1)}=\beta^{(t)}-\alpha g^{(t)} .
$$

如果 $\alpha$ 太小，每次更新幅度很小，通常比较稳定，但需要更多迭代才能接近 OLS；如果适当增大学习率，可以减少达到收敛所需的迭代次数；但如果学习率过大，就可能在最优解附近来回霞荡，甚至发散。

(ii): 因为

$$
M=\frac{n}{\text { batch size }} .
$$

所以：
$M$ 越大 → batch size 越小。
batch 越小，每次更新计算量更低、一个 epoch 内更新次数更多，但是每个 mini－batch 的梯度相对于 full gradient 有更大的随机波动。

相反：
$M$ 越小 → batch size 越大。
梯度通常更加稳定，但单次更新需要更多计算。当 $M=1$ 时，batch size 就是 $n$ ，算法实际上退化成 full－batch gradient descent。

学习率 $\alpha$ 同样控制更新步长。由于 mini－batch gradient 本身存在随机波动并且每一个batch的样本量不够大，学习率过大会让参数在最优值附近持续波动甚至不稳定；较小的学习率通常更加稳定，但收敛速度也会变慢。